# 12b – Data Imputation, Encoding & Feature Engineering


**Scope:** The missing and incorrect values are handled, categorical features are encoded.


Since we decided to predict missing and incorrect values, we have to split the training data into training and validation sets before applying imputation and encoding.

## Table of Contents
- [12b – Data Imputation, Encoding & Feature Engineering](#12b-data-imputation-encoding-feature-engineering)
- [Table of Contents](#table-of-contents)
- [1. Load processed data](#1-load-processed-data)
- [2. Train / validation / test split](#2-train-validation-test-split)
- [3. Missing value overview](#3-missing-value-overview)
- [4. Define feature types](#4-define-feature-types)
- [5. Simple imputations (median/mode)](#5-simple-imputations-medianmode)
- [5.1 Optional: RF imputation for one useful column](#51-optional-rf-imputation-for-one-useful-column)
- [6. Encoding](#6-encoding)
- [7. Feature engineering](#7-feature-engineering)
- [8. Scaling](#8-scaling)
- [9. Save](#9-save)


In [ ]:
# Table of Contents — implementation details
import os, re, math, warnings
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np
import re
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

# import onehotencoder
from sklearn.preprocessing import OneHotEncoder

# import scaler = RobustScaler()
from sklearn.preprocessing import RobustScaler

RANDOM_STATE = 42

## 1. Load processed data

In [2]:
# Load the data paths
data_dir = "../data/"

# Load the raw data into a pandas dataframe
df_train = pd.read_csv(os.path.join(data_dir, "processed_data/11_processed_train_data.csv"))
x_test = pd.read_csv(os.path.join(data_dir, "processed_data/11_processed_test_data.csv"))

# Drop the column carID from both dataframes because it is not needed for modeling
if "carID" in df_train.columns:
    df_train = df_train.drop(columns=["carID"])
if "carID" in x_test.columns:
    x_test = x_test.drop(columns=["carID"])


# Separate features and target variable from training data
print("Loaded shape:", df_train.shape)
display(df_train.head(3))

print("Loaded test shape:", x_test.shape)
display(x_test.head(3))


Loaded shape: (74467, 13)


,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,Volkswagen,Golf,2016.0,22290.0,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.0,0.0
1,Toyota,Yaris,2019.0,13790.0,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.0,0.0
2,Audi,Q2,2019.0,24990.0,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.0,0.0


Loaded test shape: (32567, 12)


,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,Hyundai,i30,NaN,Automatic,30700.0,Petrol,205.0,41.5,1.6,61.0,3.0,0.0
1,Volkswagen,Tiguan,2017.0,Semi-Auto,NaN,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
2,BMW,2 Series,2016.0,Automatic,36792.0,Petrol,125.0,51.4,1.5,94.0,2.0,0.0


## 2. Train / validation / test split

We already have a train–test split from the raw data. In this step we further split the training portion into a training and a validation set. This is essential to tune models and make design choices without touching the test set (which remains a sealed, final-evaluation set).

In [ ]:
TARGET_COL = "price"

if "split" in df_train.columns:
    train_mask = df_train["split"].eq("train")
    val_mask   = df_train["split"].eq("val")
else:
    train_mask = np.random.rand(len(df_train)) < 0.8
    val_mask   = ~train_mask

train_df = df_train.loc[train_mask].copy()
val_df   = df_train.loc[val_mask].copy()

y_train = train_df[TARGET_COL].copy()
y_val   = val_df[TARGET_COL].copy()

drop_cols = [TARGET_COL, "split"]

x_train = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
x_val   = val_df.drop(columns=[c for c in drop_cols if c in val_df.columns])
x_test  = x_test.copy()

# Final shapes
x_train.shape, x_val.shape, x_test.shape, y_train.shape, y_val.shape


((59455, 12), (15012, 12), (32567, 12), (59455,), (15012,))

## 3. Missing value overview

We first take a look at the missing values in the dataset to get an overview of the situation.

In [4]:
def missing_report(df, name):
    miss = df.isna().mean().sort_values(ascending=False)
    miss = miss[miss > 0]
    print(f"--- {name} ---")
    if miss.empty:
        print("no missing values")
    else:
        print(miss)

missing_report(x_train, "x_train")
missing_report(x_val, "x_val")
missing_report(x_test, "x_test")


--- x_train ---
mpg               0.121150
tax               0.109225
transmission      0.029400
engineSize        0.027920
previousOwners    0.025902
paintQuality%     0.024977
mileage           0.023732
model             0.022353
hasDamage         0.020469
fuelType          0.019763
year              0.004995
Brand             0.001480
dtype: float64
--- x_val ---
mpg               0.119971
tax               0.107514
transmission      0.031308
engineSize        0.026312
mileage           0.025979
paintQuality%     0.024514
previousOwners    0.023181
model             0.021583
fuelType          0.020250
hasDamage         0.020250
year              0.004063
Brand             0.001465
dtype: float64
--- x_test ---
mpg               0.117911
tax               0.106519
year              0.030921
transmission      0.029723
previousOwners    0.028741
engineSize        0.026868
mileage           0.026376
paintQuality%     0.024350
model             0.022538
fuelType          0.020143
hasDama

## 4. Define feature types

split the features into numerical and categorical features.

In [5]:
numeric_cols = x_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = x_train.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_cols[:10], categorical_cols[:10]


(['year',
  'mileage',
  'tax',
  'mpg',
  'engineSize',
  'paintQuality%',
  'previousOwners',
  'hasDamage'],
 ['Brand', 'model', 'transmission', 'fuelType'])

## 5. Simple imputations (median/mode)

For each category of features, we apply simple imputation strategies as a baseline for our modeling.

In [6]:
from sklearn.impute import SimpleImputer

# 1) Numeric
num_imputer = SimpleImputer(strategy="median")
num_imputer.fit(x_train[numeric_cols])  # only train data to prevent data leakage

x_train_imp = x_train.copy()
x_val_imp   = x_val.copy()
x_test_imp  = x_test.copy()

x_train_imp[numeric_cols] = num_imputer.transform(x_train[numeric_cols])
x_val_imp[numeric_cols]   = num_imputer.transform(x_val[numeric_cols])
x_test_imp[numeric_cols]  = num_imputer.transform(x_test[numeric_cols])

# 2) Categorical
cat_imputer = SimpleImputer(strategy="most_frequent")
cat_imputer.fit(x_train[categorical_cols])  # only train data to prevent data leakage

x_train_imp[categorical_cols] = cat_imputer.transform(x_train[categorical_cols])
x_val_imp[categorical_cols]   = cat_imputer.transform(x_val[categorical_cols])
x_test_imp[categorical_cols]  = cat_imputer.transform(x_test[categorical_cols])

missing_report(x_train_imp, "x_train_imp")
missing_report(x_val_imp, "x_val_imp")
missing_report(x_test_imp, "x_test_imp")


--- x_train_imp ---
no missing values
--- x_val_imp ---
no missing values
--- x_test_imp ---
no missing values


### 5.1 RF imputation for one useful column

Some 

In [7]:
col_to_impute = None
for cand in ["mileage", "Mileage", "km"]:
    if cand in x_train.columns and x_train[cand].isna().any():
        col_to_impute = cand
        break

if col_to_impute is not None:
    feat_cols = [c for c in x_train_imp.columns if c != col_to_impute]

    train_rows = x_train[col_to_impute].notna()
    x_rf = pd.get_dummies(x_train_imp.loc[train_rows, feat_cols], drop_first=True)
    y_rf = x_train.loc[train_rows, col_to_impute]

    rf = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(x_rf, y_rf)

    # train
    missing_train = x_train_imp[col_to_impute].isna()
    if missing_train.any():
        x_missing = pd.get_dummies(x_train_imp.loc[missing_train, feat_cols], drop_first=True)
        x_missing = x_missing.reindex(columns=x_rf.columns, fill_value=0)
        x_train_imp.loc[missing_train, col_to_impute] = rf.predict(x_missing)

    # val
    missing_val = x_val_imp[col_to_impute].isna()
    if missing_val.any():
        x_missing = pd.get_dummies(x_val_imp.loc[missing_val, feat_cols], drop_first=True)
        x_missing = x_missing.reindex(columns=x_rf.columns, fill_value=0)
        x_val_imp.loc[missing_val, col_to_impute] = rf.predict(x_missing)

    # test
    missing_test = x_test_imp[col_to_impute].isna()
    if missing_test.any():
        x_missing = pd.get_dummies(x_test_imp.loc[missing_test, feat_cols], drop_first=True)
        x_missing = x_missing.reindex(columns=x_rf.columns, fill_value=0)
        x_test_imp.loc[missing_test, col_to_impute] = rf.predict(x_missing)


## 6. Encoding

Encoding categorical features using target encoding.


In [8]:
low_card_cat = [c for c in categorical_cols if x_train_imp[c].nunique() <= 15]
high_card_cat = [c for c in categorical_cols if c not in low_card_cat]

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(x_train_imp[low_card_cat])

def apply_ohe(df):
    ohe_arr = ohe.transform(df[low_card_cat])
    ohe_cols = ohe.get_feature_names_out(low_card_cat)
    ohe_df = pd.DataFrame(ohe_arr, columns=ohe_cols, index=df.index)
    df_rest = df.drop(columns=low_card_cat)
    return pd.concat([df_rest, ohe_df], axis=1)
    

x_train_enc = apply_ohe(x_train_imp)
x_val_enc   = apply_ohe(x_val_imp)
x_test_enc  = apply_ohe(x_test_imp)


# print the columns after encoding
print("Columns after encoding:")
print(x_test_enc.columns.tolist())


# Print the number of columns after encoding for test val and train
print("Number of columns after encoding:")
print("x_train_enc:", x_train_enc.shape[1])
print("x_val_enc:", x_val_enc.shape[1])
print("x_test_enc:", x_test_enc.shape[1])

Columns after encoding:
['model', 'year', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'hasDamage', 'Brand_Audi', 'Brand_BMW', 'Brand_Ford', 'Brand_Hyundai', 'Brand_Mercedes-Benz', 'Brand_Opel', 'Brand_Toyota', 'Brand_Volkswagen', 'Brand_Škoda', 'transmission_Automatic', 'transmission_Manual', 'transmission_Other', 'transmission_Semi-Auto', 'fuelType_Diesel', 'fuelType_Electric', 'fuelType_Hybrid', 'fuelType_Other', 'fuelType_Petrol']
Number of columns after encoding:
x_train_enc: 27
x_val_enc: 27
x_test_enc: 27


## 7. Feature engineering


We add some additional features to the dataset to improve model performance.

In [9]:
CURRENT_YEAR = 2025

def add_features(df, target=None):
    df = df.copy()

    if "year" in df.columns:
        df["car_age"] = CURRENT_YEAR - df["year"]
        df.loc[df["car_age"] < 0, "car_age"] = np.nan

    if "mileage" in df.columns and "car_age" in df.columns:
        df["mileage_per_year"] = df["mileage"] / df["car_age"]
        df["mileage_per_year"] = df["mileage_per_year"].replace([np.inf, -np.inf], np.nan)

    return df

x_train_fe = add_features(x_train_enc)
x_val_fe   = add_features(x_val_enc)
x_test_fe  = add_features(x_test_enc, target=None)

x_train_fe.shape, x_val_fe.shape, x_test_fe.shape


((59455, 29), (15012, 29), (32567, 29))

## 8. Scaling

We applied a RobustScaler to the numerical features because the dataset contains outliers (price, mileage, car_age). RobustScaler uses the median and the interquartile range instead of mean and standard deviation, which makes the scaling more robust and stable. The scaler was fitted on the training set only and then applied to validation and test to avoid data leakage.”

In [10]:
scaler = RobustScaler()

# numeric cols after FE (on train)
num_after_enc = x_train_fe.select_dtypes(include=["number"]).columns

# fit on TRAIN only
scaler.fit(x_train_fe[num_after_enc])

# copy
x_train_final = x_train_fe.copy()
x_val_final   = x_val_fe.copy()
x_test_final  = x_test_fe.copy()

# scale train and val on same cols
x_train_final[num_after_enc] = scaler.transform(x_train_fe[num_after_enc])
x_val_final[num_after_enc]   = scaler.transform(x_val_fe[num_after_enc])

# for test: only the intersection of cols
test_cols = [c for c in num_after_enc if c in x_test_fe.columns]
x_test_final[test_cols] = scaler.transform(x_test_fe[test_cols])


In [11]:
# Check for NaN values in each dataset
def check_nan(df, name):
    nan_cols = df.columns[df.isna().any()].tolist()
    if nan_cols:
        print(f"{name} has NaN values in columns: {nan_cols}")
    else:
        print(f"{name} has no NaN values.")


check_nan(x_train_final, "x_train_final")
check_nan(y_train.to_frame(), "y_train")
check_nan(x_val_final, "x_val_final")
check_nan(y_val.to_frame(), "y_val")
check_nan(x_test_final, "x_test_final")

x_train_final has no NaN values.
y_train has no NaN values.
x_val_final has no NaN values.
y_val has no NaN values.
x_test_final has no NaN values.


## 9. Save

In [12]:
# Save Processed Datasets
"""
Save x_train, y_train, x_val, y_val, and x_test separately.
This structure is cleaner for later model loading and avoids re-splitting.
"""

output_dir = os.path.join(data_dir, "encoded_data")
os.makedirs(output_dir, exist_ok=True)

# Save feature and target sets separately
x_train_final.to_csv(os.path.join(output_dir, "12_x_train.csv"), index=False)
y_train.to_csv(os.path.join(output_dir, "12_y_train.csv"), index=False)

x_val_final.to_csv(os.path.join(output_dir, "12_x_val.csv"), index=False)
y_val.to_csv(os.path.join(output_dir, "12_y_val.csv"), index=False)

x_test_final.to_csv(os.path.join(output_dir, "12_x_test.csv"), index=False)

print("Processed data saved successfully (x/y separated):")
print(f"x_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"x_val:   {x_val.shape}, y_val: {y_val.shape}")
print(f"x_test:  {x_test.shape}")

Processed data saved successfully (x/y separated):
x_train: (59455, 12), y_train: (59455,)
x_val:   (15012, 12), y_val: (15012,)
x_test:  (32567, 12)
